# Teen Mental Health: Social Media, Sleep, and Depression
## Reproducible Research Notebook

A study of 1,200 U.S. teenagers examining three research questions:
- **RQ1**: Which behavioral/lifestyle factors best predict depression risk?
- **RQ2**: Is there a dose-response between social media use and mental health outcomes?
- **RQ3**: Does sleep mediate the screen time → mental health relationship?

**Dataset**: Teen_Mental_Health_Dataset.csv (N=1,200, 13 variables)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 150
sns.set_style('whitegrid')
sns.set_palette('husl')


In [ ]:
import os

# Determine paths relative to the notebook location
_nb_dir = os.path.dirname(os.path.abspath('research_notebook.ipynb'))
_data_path = os.path.join(_nb_dir, '..', 'Teen_Mental_Health_Dataset.csv')
_out_dir = _nb_dir  # figures saved next to notebook

df = pd.read_csv(_data_path)
print(f"Dataset shape: {df.shape}")
print(f"\nDepression prevalence: {df['depression_label'].mean()*100:.1f}%")
df.describe().round(2)


## Research Question 1: Predictors of Depression


In [ ]:
# ── Encode categoricals ──────────────────────────────────────────────────────
df['gender_enc'] = df['gender'].map({'male': 0, 'female': 1})
df['platform_enc'] = df['platform_usage'].map({'Instagram': 0, 'TikTok': 1, 'Both': 2})
df['interaction_enc'] = df['social_interaction_level'].map({'low': 0, 'medium': 1, 'high': 2})

feature_cols = [
    'age', 'gender_enc', 'daily_social_media_hours', 'platform_enc',
    'sleep_hours', 'screen_time_before_sleep', 'academic_performance',
    'physical_activity', 'interaction_enc', 'stress_level',
    'anxiety_level', 'addiction_level'
]

X = df[feature_cols].values
y = df['depression_label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Logistic Regression ──────────────────────────────────────────────────────
clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_scaled, y)

y_pred = clf.predict(X_scaled)
y_prob = clf.predict_proba(X_scaled)[:, 1]

auc = roc_auc_score(y, y_prob)
acc = accuracy_score(y, y_pred)

# McFadden R2
from scipy.special import expit
log_loss_null = -np.sum(y * np.log(y.mean()) + (1 - y) * np.log(1 - y.mean()))
log_loss_model = -np.sum(y * np.log(np.clip(y_prob, 1e-15, 1 - 1e-15)) +
                         (1 - y) * np.log(np.clip(1 - y_prob, 1e-15, 1 - 1e-15)))
mcfadden_r2 = 1 - log_loss_model / log_loss_null

print(f"AUC:          {auc:.4f}")
print(f"Accuracy:     {acc*100:.2f}%")
print(f"McFadden R²:  {mcfadden_r2:.3f}")

# ── Coefficients & Odds Ratios ───────────────────────────────────────────────
coef_df = pd.DataFrame({
    'feature':      feature_cols,
    'coefficient':  clf.coef_[0],
    'odds_ratio':   np.exp(clf.coef_[0])
}).sort_values('coefficient', key=abs, ascending=False).reset_index(drop=True)

print("\nTop feature coefficients and odds ratios:")
print(coef_df.round(4).to_string())


In [ ]:
# ── Figure 1: Feature Importance ─────────────────────────────────────────────
sorted_df = coef_df.sort_values('coefficient', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))

colors = [plt.cm.RdYlGn_r(0.85) if c > 0 else plt.cm.RdYlGn_r(0.15)
          for c in sorted_df['coefficient']]

bars = ax.barh(sorted_df['feature'], sorted_df['coefficient'].abs(),
               color=colors, edgecolor='white', linewidth=0.5)

ax.set_xlabel('|Coefficient| (standardised)', fontsize=11)
ax.set_title('Figure 1: Logistic Regression Feature Importance for Depression Prediction',
             fontsize=11, fontweight='bold', pad=10)

# Legend patches
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=plt.cm.RdYlGn_r(0.85), label='Risk factor (+coef)'),
                   Patch(facecolor=plt.cm.RdYlGn_r(0.15), label='Protective factor (-coef)')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(_out_dir, 'fig1_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(_out_dir, 'fig1_feature_importance.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure 2: Correlation Heatmap ─────────────────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 7}, linewidths=0.4)
ax.set_title('Figure 2: Correlation Matrix of Mental Health Indicators',
             fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(_out_dir, 'fig2_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(_out_dir, 'fig2_correlation_heatmap.pdf'), bbox_inches='tight')
plt.show()


## Research Question 2: Social Media Dose-Response


In [ ]:
# ── Bin social media hours ───────────────────────────────────────────────────
df['sm_bin'] = pd.cut(df['daily_social_media_hours'],
                      bins=[0, 3, 5, 8],
                      labels=['Low (1-3h)', 'Medium (3-5h)', 'High (5-8h)'],
                      right=True)

bin_stats = df.groupby('sm_bin', observed=True)[['stress_level', 'anxiety_level', 'addiction_level']].mean()
print("Mean outcomes by social media bin:")
print(bin_stats.round(3))

# ── Pearson correlations ─────────────────────────────────────────────────────
print("\nPearson correlations with daily_social_media_hours:")
for col in ['stress_level', 'anxiety_level', 'addiction_level', 'sleep_hours']:
    r, p = stats.pearsonr(df['daily_social_media_hours'], df[col])
    print(f"  {col:35s}: r={r:+.4f}, p={p:.4f}")

# ── One-way ANOVA ────────────────────────────────────────────────────────────
groups_stress  = [grp['stress_level'].values  for _, grp in df.groupby('sm_bin', observed=True)]
groups_anxiety = [grp['anxiety_level'].values for _, grp in df.groupby('sm_bin', observed=True)]

f_stress,  p_stress  = stats.f_oneway(*groups_stress)
f_anxiety, p_anxiety = stats.f_oneway(*groups_anxiety)

print(f"\nANOVA stress_level  by sm_bin: F={f_stress:.4f},  p={p_stress:.4f}")
print(f"ANOVA anxiety_level by sm_bin: F={f_anxiety:.4f}, p={p_anxiety:.4f}")


In [ ]:
# ── Figure 3: Platform Stress / Anxiety ──────────────────────────────────────
platform_stats = df.groupby('platform_usage')[['stress_level', 'anxiety_level']].mean().reset_index()
platform_order = ['Instagram', 'TikTok', 'Both']
platform_stats = platform_stats.set_index('platform_usage').loc[platform_order].reset_index()

x = np.arange(len(platform_order))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, platform_stats['stress_level'],  width, label='Stress Level',  color='#E07070')
bars2 = ax.bar(x + width/2, platform_stats['anxiety_level'], width, label='Anxiety Level', color='#70A0E0')

ax.set_xticks(x)
ax.set_xticklabels(platform_order, fontsize=11)
ax.set_ylabel('Mean Score', fontsize=11)
ax.set_title('Figure 3: Mean Stress and Anxiety by Social Media Platform',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, max(platform_stats[['stress_level', 'anxiety_level']].values.max() * 1.2, 1))

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(_out_dir, 'fig3_platform_stress_anxiety.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(_out_dir, 'fig3_platform_stress_anxiety.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure 4: Dose-Response Line Plot ────────────────────────────────────────
dose_df = df.groupby('sm_bin', observed=True)[['stress_level', 'anxiety_level', 'addiction_level']].mean()

fig, ax = plt.subplots(figsize=(8, 5))
markers = ['o', 's', '^']
colors  = ['#E07070', '#70A0E0', '#70C870']

for col, marker, color in zip(['stress_level', 'anxiety_level', 'addiction_level'],
                               markers, colors):
    ax.plot(dose_df.index.astype(str), dose_df[col],
            marker=marker, color=color, linewidth=2, markersize=8,
            label=col.replace('_', ' ').title())

ax.set_xlabel('Social Media Usage Bin', fontsize=11)
ax.set_ylabel('Mean Score', fontsize=11)
ax.set_title('Figure 4: Dose-Response of Social Media Use on Mental Health Outcomes',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.tick_params(axis='x', labelsize=10)

plt.tight_layout()
plt.savefig(os.path.join(_out_dir, 'fig4_dose_response.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(_out_dir, 'fig4_dose_response.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure 5: Scatter Matrix ──────────────────────────────────────────────────
outcome_vars = ['stress_level', 'anxiety_level', 'addiction_level', 'sleep_hours']
x_var = 'daily_social_media_hours'

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

scatter_colors = ['#E07070', '#70A0E0', '#70C870', '#C070E0']

for i, (col, color) in enumerate(zip(outcome_vars, scatter_colors)):
    ax = axes[i]
    x_vals = df[x_var].values
    y_vals = df[col].values

    ax.scatter(x_vals, y_vals, alpha=0.25, color=color, s=15, edgecolors='none')

    # Regression trend line
    z = np.polyfit(x_vals, y_vals, 1)
    p_line = np.poly1d(z)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
    ax.plot(x_line, p_line(x_line), color='black', linewidth=1.8, linestyle='--')

    r, p_val = stats.pearsonr(x_vals, y_vals)
    ax.set_xlabel(x_var.replace('_', ' ').title(), fontsize=10)
    ax.set_ylabel(col.replace('_', ' ').title(), fontsize=10)
    ax.set_title(f'{col.replace("_", " ").title()}\n(r={r:.3f}, p={p_val:.3f})', fontsize=10)

fig.suptitle('Figure 5: Social Media Hours vs Mental Health Outcomes',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(_out_dir, 'fig5_scatter_matrix.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(_out_dir, 'fig5_scatter_matrix.pdf'), bbox_inches='tight')
plt.show()


## Research Question 3: Sleep as a Mediator


In [ ]:
# ── Baron & Kenny Mediation Analysis ─────────────────────────────────────────
# X = screen_time_before_sleep, M = sleep_hours
# Y candidates: stress_level, anxiety_level
# Controls: age, gender_enc, physical_activity

controls = ['age', 'gender_enc', 'physical_activity']
X_med = df['screen_time_before_sleep'].values.reshape(-1, 1)
M_med = df['sleep_hours'].values.reshape(-1, 1)
controls_arr = df[controls].values

mediation_rows = []

for outcome in ['stress_level', 'anxiety_level']:
    Y = df[outcome].values.reshape(-1, 1)

    # Path c: X → Y (total effect, controlling for covariates)
    Xc_c = np.hstack([X_med, controls_arr])
    lr_c = LinearRegression().fit(Xc_c, Y.ravel())
    c_coef = lr_c.coef_[0]

    # Path a: X → M (controlling for covariates)
    Xc_a = np.hstack([X_med, controls_arr])
    lr_a = LinearRegression().fit(Xc_a, M_med.ravel())
    a_coef = lr_a.coef_[0]

    # Path b & c': X + M → Y (controlling for covariates)
    Xc_bc = np.hstack([X_med, M_med, controls_arr])
    lr_bc = LinearRegression().fit(Xc_bc, Y.ravel())
    c_prime = lr_bc.coef_[0]   # direct effect of X
    b_coef  = lr_bc.coef_[1]   # effect of M

    indirect = a_coef * b_coef

    mediation_rows.append({
        'outcome':        outcome,
        'path_c_total':   round(c_coef,    4),
        'path_a_X_M':     round(a_coef,    4),
        'path_b_M_Y':     round(b_coef,    4),
        'path_c_direct':  round(c_prime,   4),
        'indirect_a_x_b': round(indirect,  4)
    })

mediation_df = pd.DataFrame(mediation_rows)
print("Baron & Kenny Mediation Summary:")
print(mediation_df.to_string(index=False))

# Pearson r for path a
r_a, p_a = stats.pearsonr(df['screen_time_before_sleep'], df['sleep_hours'])
print(f"\nPath a Pearson r (screen_time → sleep): r={r_a:.3f}, p={p_a:.4f}")

# ── Sleep bins ────────────────────────────────────────────────────────────────
df['sleep_bin'] = pd.cut(df['sleep_hours'],
                         bins=[0, 6, 8, 24],
                         labels=['Short (<6h)', 'Adequate (6-8h)', 'Long (>8h)'],
                         right=True)

sleep_bin_stats = df.groupby('sleep_bin', observed=True).agg(
    n=('sleep_hours', 'count'),
    mean_stress=('stress_level', 'mean'),
    mean_anxiety=('anxiety_level', 'mean'),
    depression_rate=('depression_label', 'mean')
).round(4)

print("\nStats by sleep bin:")
print(sleep_bin_stats.to_string())


In [ ]:
# ── Figure 6: Mediation Scatter (3 panels) ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

panels = [
    ('screen_time_before_sleep', 'sleep_hours',    'Screen Time → Sleep'),
    ('sleep_hours',              'stress_level',   'Sleep → Stress'),
    ('sleep_hours',              'anxiety_level',  'Sleep → Anxiety'),
]
panel_colors = ['#7090D0', '#E07070', '#70C8A0']

for ax, (xc, yc, title), color in zip(axes, panels, panel_colors):
    xv = df[xc].values
    yv = df[yc].values

    ax.scatter(xv, yv, alpha=0.25, color=color, s=12, edgecolors='none')

    z = np.polyfit(xv, yv, 1)
    p_line = np.poly1d(z)
    xl = np.linspace(xv.min(), xv.max(), 200)
    ax.plot(xl, p_line(xl), color='black', linewidth=2, linestyle='--')

    r, p_val = stats.pearsonr(xv, yv)
    ax.set_xlabel(xc.replace('_', ' ').title(), fontsize=10)
    ax.set_ylabel(yc.replace('_', ' ').title(), fontsize=10)
    ax.set_title(f'{title}\n(r={r:.3f}, p={p_val:.3f})', fontsize=10)

fig.suptitle('Figure 6: Mediation Pathway — Screen Time → Sleep → Mental Health',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(_out_dir, 'fig6_mediation_scatter.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(_out_dir, 'fig6_mediation_scatter.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure 7: Sleep Bins Bar Chart ───────────────────────────────────────────
sleep_plot = df.groupby('sleep_bin', observed=True)[['stress_level', 'anxiety_level']].mean().reset_index()
bin_order  = ['Short (<6h)', 'Adequate (6-8h)', 'Long (>8h)']
sleep_plot = sleep_plot.set_index('sleep_bin').loc[bin_order].reset_index()

x = np.arange(len(bin_order))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, sleep_plot['stress_level'],  width, label='Stress Level',  color='#E07070')
bars2 = ax.bar(x + width/2, sleep_plot['anxiety_level'], width, label='Anxiety Level', color='#70A0E0')

ax.set_xticks(x)
ax.set_xticklabels(bin_order, fontsize=11)
ax.set_ylabel('Mean Score', fontsize=11)
ax.set_title('Figure 7: Mental Health Outcomes by Sleep Duration Category',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, max(sleep_plot[['stress_level', 'anxiety_level']].values.max() * 1.2, 1))

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(_out_dir, 'fig7_sleep_bins.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(_out_dir, 'fig7_sleep_bins.pdf'), bbox_inches='tight')
plt.show()


## Summary of Key Findings

### RQ1 — Predictors of Depression
- The logistic regression model achieved AUC = 0.9936, Accuracy = 98.75%, McFadden R² = 0.688
- Top predictors: sleep_hours (OR=0.111, protective), anxiety_level (OR=6.50), stress_level (OR=5.68), daily_social_media_hours (OR=5.18)
- Only sleep_hours, daily_social_media_hours, stress_level, and anxiety_level showed significant bivariate correlations with depression

### RQ2 — Social Media Dose-Response
- No statistically significant dose-response detected (Pearson r ≈ 0.03, all p > 0.28; ANOVA p > 0.06)
- Platform differences: TikTok users show highest anxiety (mean=5.75), multi-platform users show highest stress (mean=5.55)
- Stress and anxiety levels remain relatively flat across Low/Medium/High usage bins

### RQ3 — Sleep Mediation
- Baron & Kenny mediation not supported: path a (screen_time→sleep) is non-significant (r=0.010, p=0.72)
- However, sleep duration strongly predicts depression specifically: all 31 depression cases occur in the Short sleep (<6h) group
- Depression prevalence: Short sleep = 6.5%, Adequate sleep = 0%, Long sleep = 0%
